# OpenRouter Prompt Tester (Qwen + Other SLMs)

Use this notebook to manually edit **System** and **User** prompts and test any OpenRouter model.

## Quick start
1. Set `OPENROUTER_API_KEY` in your environment (or paste it into the config cell).
2. Optionally run the model listing cell to discover available model IDs.
3. Edit `MODEL_KEY`, `SYSTEM_PROMPT`, and `USER_PROMPT`.
4. Run the request cell.


In [1]:
import os
import json
import requests
from typing import List, Dict, Any

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

# Optional headers (recommended by OpenRouter).
HTTP_REFERER = os.getenv("OPENROUTER_HTTP_REFERER", "http://localhost")
X_TITLE = os.getenv("OPENROUTER_X_TITLE", "OpenRouter Prompt Tester")

if not OPENROUTER_API_KEY:
    print("OPENROUTER_API_KEY is empty. Set it in env or assign OPENROUTER_API_KEY in this cell.")

def _headers(api_key: str) -> Dict[str, str]:
    return {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": HTTP_REFERER,
        "X-Title": X_TITLE,
    }


In [2]:
def list_models(api_key: str, search: str = "qwen", limit: int = 30) -> List[Dict[str, Any]]:
    """Fetch OpenRouter models and optionally filter by text (e.g., 'qwen')."""
    r = requests.get(f"{OPENROUTER_BASE_URL}/models", headers=_headers(api_key), timeout=60)
    r.raise_for_status()
    data = r.json().get("data", [])
    if search:
        s = search.lower()
        data = [m for m in data if s in m.get("id", "").lower()]
    data = data[:limit]
    for i, m in enumerate(data, 1):
        print(f"{i:2d}. {m.get('id')}")
    return data

# Example: show Qwen models
# _ = list_models(OPENROUTER_API_KEY, search="qwen", limit=30)


## Single-turn test
Edit the prompt variables below, then run the request cell.

In [3]:
# OpenRouter model IDs for your shortlist.
GEMMA_2_9B_IT = "google/gemma-2-9b-it"
GEMMA_3_4B_IT = "google/gemma-3-4b-it"
MISTRAL_7B = "mistralai/mistral-7b-instruct"
QWEN_2_5_7B = "qwen/qwen-2.5-7b-instruct"
LLAMA3_8B = "meta-llama/llama-3-8b-instruct"

MODEL_OPTIONS = {
    "gemma-2-9b-it": GEMMA_2_9B_IT,
    "gemma-3-4b-it": GEMMA_3_4B_IT,
    "mistral:7b": MISTRAL_7B,
    "qwen2.5:7b": QWEN_2_5_7B,
    "llama3:8b": LLAMA3_8B,
}

# Switch this key to choose the model quickly.
MODEL_KEY = "gemma-2-9b-it"
MODEL = MODEL_OPTIONS[MODEL_KEY]

# SYSTEM_PROMPT = """
# "You are solving a calendar scheduling task using a fixed Tree-of-Thought path. "
# "Current level - Level 4: Final answer and verification"
# "Level goal: produce properly formatted output
# "Return only the intermediate result for the thought provided in the user input "
# "Do not provide final solution at this step."
# """

SYSTEM_PROMPT = """
You are a calendar scheduling solver.
Produce one final answer using format exactly:
Here is the proposed time: <Day>, <HH:MM> - <HH:MM>.
Return only one line.
"""


USER_PROMPT = """

You are an expert at scheduling meetings. You are given a few constraints on the existing schedule of each participant, the meeting duration, and possibly some preferences on the meeting time. Note there exists a solution that works with existing schedule of every participant. Here are a few example tasks and solutions:
TASK: You need to schedule a meeting for Roger, Karen and Dorothy for half an hour between the work hours of 9:00 to 17:00 on Monday.
Here are the existing schedules for everyone during the day:
Roger's calendar is wide open the entire day.
Karen has meetings on Monday during 10:00 to 10:30, 11:30 to 12:00, 12:30 to 13:00, 14:00 to 15:00, 15:30 to 16:00.
Dorothy is busy on Monday during 9:00 to 10:00, 10:30 to 11:00, 11:30 to 12:00, 12:30 to 13:00, 14:00 to 15:30, 16:00 to 17:00.
You would like to schedule the meeting at their earliest availability.
Find a time that works for everyone's schedule and constraints.
SOLUTION: Here is the proposed time: Monday, 11:00 - 11:30

TASK: You need to schedule a meeting for Douglas, Lawrence and Isabella for half an hour between the work hours of 9:00 to 17:00 on Monday.
Here are the existing schedules for everyone during the day:
Douglas has meetings on Monday during 12:00 to 12:30, 15:00 to 15:30.
Lawrence has meetings on Monday during 10:30 to 12:00, 13:00 to 13:30, 14:00 to 14:30, 15:30 to 16:00.
Isabella is busy on Monday during 9:00 to 12:30, 13:30 to 17:00.
Find a time that works for everyone's schedule and constraints.
SOLUTION: Here is the proposed time: Monday, 12:30 - 13:00

TASK: You need to schedule a meeting for Joshua, Denise and Jeremy for one hour between the work hours of 9:00 to 17:00 on Monday.
Here are the existing schedules for everyone during the day:
Joshua is busy on Monday during 10:00 to 10:30, 12:00 to 12:30, 14:00 to 14:30, 15:00 to 15:30.
Denise's calendar is wide open the entire day.
Jeremy has meetings on Monday during 9:30 to 10:30, 12:00 to 13:00, 13:30 to 14:00, 14:30 to 15:00, 15:30 to 16:00, 16:30 to 17:00.
Find a time that works for everyone's schedule and constraints.
SOLUTION: Here is the proposed time: Monday, 10:30 - 11:30

TASK: You need to schedule a meeting for Alan, Elizabeth and Denise for half an hour between the work hours of 9:00 to 17:00 on Monday.
Here are the existing schedules for everyone during the day:
Alan has blocked their calendar on Monday during 13:00 to 13:30, 14:00 to 14:30.
Elizabeth is busy on Monday during 9:00 to 9:30, 11:00 to 11:30.
Denise has blocked their calendar on Monday during 9:00 to 10:00, 10:30 to 11:30, 12:30 to 13:00, 13:30 to 14:00, 14:30 to 16:00, 16:30 to 17:00.
You would like to schedule the meeting at their earliest availability.
Find a time that works for everyone's schedule and constraints.
SOLUTION: Here is the proposed time: Monday, 10:00 - 10:30

TASK: You need to schedule a meeting for Mason, Bruce and Christopher for half an hour between the work hours of 9:00 to 17:00 on Monday.
Here are the existing schedules for everyone during the day:
Mason has meetings on Monday during 9:30 to 10:00, 11:00 to 11:30, 14:30 to 15:00, 16:30 to 17:00.
Bruce is free the entire day.
Christopher is busy on Monday during 9:30 to 10:30, 11:30 to 12:30, 15:00 to 17:00.
Mason would rather not meet on Monday before 12:30.
Find a time that works for everyone's schedule and constraints.
SOLUTION: Here is the proposed time: Monday, 12:30 - 13:00

TASK: You need to schedule a meeting for Raymond, Billy and Donald for half an hour between the work hours of 9:00 to 17:00 on Monday.
Here are the existing schedules for everyone during the day:
Raymond has blocked their calendar on Monday during 9:00 to 9:30, 11:30 to 12:00, 13:00 to 13:30, 15:00 to 15:30.
Billy has meetings on Monday during 10:00 to 10:30, 12:00 to 13:00, 16:30 to 17:00.
Donald has meetings on Monday during 9:00 to 9:30, 10:00 to 11:00, 12:00 to 13:00, 14:00 to 14:30, 16:00 to 17:00.
Billy would like to avoid more meetings on Monday after 15:00.

Thought 1 -
Extract duration, days, work hours, and preferences first.

Intermediate Result for thought 1-
* **Duration:** 30 minutes
* **Day:** Monday
* **Work hours:** 9:00 to 17:00
* **Preferences:**  Billy would like to avoid more meetings on Monday after 15:00. 

Thought 2 -
Normalize first by day, then group by participant.

Intermediate Result for thought 2-
* **Monday:** 
    * Raymond: 9:00-9:30, 11:30-12:00, 13:00-13:30, 15:00-15:30 blocked
    * Billy: 10:00-10:30, 12:00-13:00, 16:30-17:00 blocked 
    * Donald: 9:00-9:30, 10:00-11:00, 12:00-13:00, 14:00-14:30, 16:00-17:00 blocked 

Thought 3 -
Compute each participant's free intervals, then find the common slots where all participants are available. Add very short explanation.

Intermediate Result for thought 3-
* **Raymond:** 9:30-11:30, 12:00-13:00, 13:30-15:00, 15:30-17:00
* **Billy:** 9:30-10:00, 10:30-12:00, 13:00-16:30 
* **Donald:** 9:30-10:00, 11:00-12:00, 13:30-14:00, 14:30-16:00 

Finding the overlapping free slots within these intervals will reveal the possible meeting times.

Thought 4-
Prefer a clearly safe valid candidate with buffer from adjacent busy intervals, and avoid edge-touching slots unless necessary.

Intermediate Result for thought 4-
Considering Billy's preference to avoid meetings after 15:00,  the most suitable candidate is  **Monday, 12:30 - 13:00**. This slot offers a buffer from Raymond's and Donald's previous commitments and aligns with Billy's desire to finish meetings before 15:00.

Thought 5-
Verify day, time, duration, and output format and then return the best slot. Add very short explanation.

Intermediate Result for thought 5-
Monday, 12:30 - 13:00 This slot works for all participants and respects Billy's preference to avoid meetings after 15:00. 

Latest Thought-
Verify day, start time, end time, duration, and final output format before returning the final answer.

Provide a concise result for the latest thought only, for the last unsolved task.

"""

TEMPERATURE = 0.2
MAX_TOKENS = 400
TOP_P = 1.0

MESSAGES = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]


In [4]:
def chat_completion(
    api_key: str,
    model: str,
    messages: List[Dict[str, str]],
    temperature: float = 0.2,
    max_tokens: int = 400,
    top_p: float = 1.0,
    extra_payload: Dict[str, Any] | None = None,
) -> Dict[str, Any]:
    payload: Dict[str, Any] = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": top_p,
    }
    if extra_payload:
        payload.update(extra_payload)

    r = requests.post(
        f"{OPENROUTER_BASE_URL}/chat/completions",
        headers=_headers(api_key),
        json=payload,
        timeout=120,
    )

    if r.status_code >= 400:
        print("Request failed:", r.status_code)
        try:
            print(json.dumps(r.json(), indent=2))
        except Exception:
            print(r.text)
        r.raise_for_status()

    return r.json()


In [5]:
response = chat_completion(
    api_key=OPENROUTER_API_KEY,
    model=MODEL,
    messages=MESSAGES,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    top_p=TOP_P,
)

assistant_text = response["choices"][0]["message"]["content"]
print(assistant_text)

print("\n--- usage ---")
print(json.dumps(response.get("usage", {}), indent=2))


Request failed: 404
{
  "error": {
    "message": "No endpoints found for google/gemma-2-9b-it.",
    "code": 404
  },
  "user_id": "user_38lQvyRfb8dabjweyBtgpW9mUxd"
}


HTTPError: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions

## Multi-turn test (optional)
Append follow-up messages manually and rerun.

In [ ]:
history = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

# First response
resp1 = chat_completion(OPENROUTER_API_KEY, MODEL, history, TEMPERATURE, MAX_TOKENS, TOP_P)
a1 = resp1["choices"][0]["message"]["content"]
history.append({"role": "assistant", "content": a1})
print("Assistant (turn 1):\n", a1)

# Follow-up user message (edit this)
FOLLOWUP_USER_PROMPT = "Please shorten that plan to 3 steps and add one risk per step."
history.append({"role": "user", "content": FOLLOWUP_USER_PROMPT})

resp2 = chat_completion(OPENROUTER_API_KEY, MODEL, history, TEMPERATURE, MAX_TOKENS, TOP_P)
a2 = resp2["choices"][0]["message"]["content"]
print("\nAssistant (turn 2):\n", a2)
